# Accounting section

## Transformer accounting

(a) Consider a GPT-2 XL-sized model using our assignment architecture, which has the following configuration:

```yaml
vocab_size: 50,257
context_length: 1,024
num_layers: 48
d_model: 1,600
num_heads: 25
d_ff: 4,288 (the nearest multiple of 64 to 8/3 × 1, 600)
```

Suppose we constructed our model using this configuration.

1) How many trainable parameters would our model have? 

2) Assuming each parameter is represented using single-precision floating point, how much memory is required to just load this model?

**Deliverable**: A one-to-two sentence response.


In [1]:
vocab_size = 50257
context_length = 1024
num_layers = 48
d_model = 1600
num_heads = 25
d_ff = 4288

In [2]:
# 1. Trainable parameters ()

# 1.1 Outside Transformer Block parameters
embed_params = vocab_size * d_model
rms_norm_after_transformer_block = d_model
linear_layer_final = d_model * vocab_size

# 1.2 Tranformer Block parameters
rms_norm_params_inside_transformer_block = d_model * 2

# 1.2.1 MutliHead Attention
q_proj = d_model * d_model
k_proj = d_model * d_model
v_proj = d_model * d_model
o_proj = d_model * d_model
multi_head_attention = q_proj + k_proj + v_proj + o_proj

# 1.2.2. SwiGLU
swiglu_matricies = d_ff * d_model * 3

# 1.3 General Result
trainable_parameters_tranformer_blocks = (
    (multi_head_attention + rms_norm_params_inside_transformer_block + swiglu_matricies) * num_layers
)
trainable_parameters = (
    embed_params + rms_norm_after_transformer_block + linear_layer_final + trainable_parameters_tranformer_blocks
)
print(f"1) {trainable_parameters} parameters (without weight tying)")

# 2. How much memory is required to just load this model?

print(f"2) {(trainable_parameters * 4 / 1024 / 1024 / 1024):.6f} Gb is required to train this model")


1) 1640452800 parameters (without weight tying)
2) 6.111163 Gb is required to train this model


(b) Identify the matrix multiplies required to complete a forward pass of our GPT-2 XL-shaped model. How many FLOPs do these matrix multiplies require in total? Assume that our input sequence has context_length tokens.

**Deliverable**: A list of matrix multiplies (with descriptions), and the total number of FLOPs
required.

> Rule: Given 𝐴 ∈ ℝ𝑚×𝑛 and 𝐵 ∈ ℝ𝑛×𝑝, the matrix-matrix product 𝐴𝐵 requires 2𝑚𝑛𝑝 FLOPs.

To start with, it woud be helpful to list out all matrix multiplication in our architecture.

From the previous exercise we have parameters and hence enough information to calculate FLOPs operations.

For every TransformerBlock we have next matrix multiplication parts:

1) MultiHead attention:
```python
q_proj = 2 * d_model * d_model * context_length
k_proj = 2 * d_model * d_model * context_length
v_proj = 2 * d_model * d_model * context_length
o_proj = 2 * d_model * d_model * context_length
qkv = 4 * d_model * context_length * context_length
mha = q_proj + k_proj + v_proj + o_proj + qkv
```

2) SwiGLU:
```python
w1 = 2 * d_ff * d_model * context_length
w2 = 2 * d_ff * d_model * context_length
w3 = 2 * d_ff * d_model * context_length
swiglu = w1 + w2 + w3
```

Since then we have next formula for one TransformerBlock:

```python
final_layer = d_model * vocab_size * context_length * 2
transformer_block_flops = (mha + swiglu) * num_layers + final_layer
```



In [3]:
q_proj = 2 * d_model * d_model * context_length
k_proj = 2 * d_model * d_model * context_length
v_proj = 2 * d_model * d_model * context_length
o_proj = 2 * d_model * d_model * context_length
qkv = 4 * d_model * context_length * context_length
mha = q_proj + k_proj + v_proj + o_proj + qkv

w1 = 2 * d_ff * d_model * context_length
w2 = 2 * d_ff * d_model * context_length
w3 = 2 * d_ff * d_model * context_length
swiglu = w1 + w2 + w3

final_layer = d_model * vocab_size * context_length * 2
transformer_block_flops = (mha + swiglu) * num_layers + final_layer
transformer_block_flops

3516769894400

(c) Based on your analysis above, which parts of the model require the most FLOPs?

To figure this out, I need to deep dive into each layer:

In [4]:
print(
f"""
MHA layer: {mha} FLOPs
Swiglu layer: {swiglu} FLOPs
Total Transformer Block: {transformer_block_flops} FLOPs
""")


MHA layer: 27682406400 FLOPs
Swiglu layer: 42152755200 FLOPs
Total Transformer Block: 3516769894400 FLOPs



And, of course, the bottleneck of our system is `MHA` layer, because in addition to `QKV` projection that takes sort of substantial computations we have attention mechanism that mostly dependes on a `context length` and especially this formula shows us quadratic computational complexity.


(d) Repeat your analysis with GPT-2 small (12 layers, 768 d_model, 12 heads), GPT-2 medium
(24 layers, 1024 d_model, 16 heads), and GPT-2 large (36 layers, 1280 d_model, 20 heads). As
the model size increases, which parts of the Transformer LM take up proportionally more or
less of the total FLOPs?
Deliverable: For each model, provide a breakdown of model components and its associated
FLOPs (as a proportion of the total FLOPs required for a forward pass). In addition, provide
a one-to-two sentence description of how varying the model size changes the proportional
FLOPs of each component.

In [5]:
def flops_analysis(
        d_model: int, d_ff: int, num_layers: int, 
        context_length: int = context_length, vocab_size: int = vocab_size, 
) -> None:
    q_proj = 2 * d_model * d_model * context_length
    k_proj = 2 * d_model * d_model * context_length
    v_proj = 2 * d_model * d_model * context_length
    o_proj = 2 * d_model * d_model * context_length
    qkv = 4 * d_model * context_length * context_length
    mha = q_proj + k_proj + v_proj + o_proj + qkv

    w1 = 2 * d_ff * d_model * context_length
    w2 = 2 * d_ff * d_model * context_length
    w3 = 2 * d_ff * d_model * context_length
    swiglu = w1 + w2 + w3

    final_layer = d_model * vocab_size * context_length * 2
    transformer_block_flops = (mha + swiglu) * num_layers + final_layer

    print(
f"""
MHA layer: {mha} FLOPs
Swiglu layer: {swiglu} FLOPs
Total Transformer Block: {transformer_block_flops} FLOPs

MHA/Swiglu: {mha / swiglu}

Swiglu/MHA: {swiglu / mha}
""")

In [6]:
models = [
    (
        "GPT-2 small",
        {
            "num_layers": 12,
            "d_model": 768,
            "d_ff": 768 * 8 // 3,
        }
    ),
    (
        "GPT-2 medium",
        {
            "num_layers": 24,
            "d_model": 1024,
            "d_ff": 2752,
        }
    ),
    (
        "GPT-2 large",
        {
            "num_layers": 36,
            "d_model": 1280,
            "d_ff": 3456,
        }
    )
]

for model in models:
    print("=" * 10 + model[0] + "=" * 10)
    flops_analysis(**model[1])

==========GPT-2 small==========

MHA layer: 8053063680 FLOPs
Swiglu layer: 9663676416 FLOPs
Total Transformer Block: 291648307200 FLOPs

MHA/Swiglu: 0.8333333333333334

Swiglu/MHA: 1.2

==========GPT-2 medium==========

MHA layer: 12884901888 FLOPs
Swiglu layer: 17314086912 FLOPs
Total Transformer Block: 830172299264 FLOPs

MHA/Swiglu: 0.7441860465116279

Swiglu/MHA: 1.34375

==========GPT-2 large==========

MHA layer: 18790481920 FLOPs
Swiglu layer: 27179089920 FLOPs
Total Transformer Block: 1786650296320 FLOPs

MHA/Swiglu: 0.691358024691358

Swiglu/MHA: 1.4464285714285714



A little analysis:

For given context length and vocab size, as the model size increases, as less ration of attention mechanism and swiglu layer. So, it means that our computations are less dependent on a input size.

(e) Take GPT-2 XL and increase the context length to 16,384. How does the total FLOPs for one
forward pass change? How does the relative contribution of FLOPs of the model components
change?

In [7]:
print("GPT-2 XL with context_length=1024")
flops_analysis(
        d_model, d_ff, num_layers, 
        1024, vocab_size, 
)
print("=" * 50)
print("GPT-2 XL with context_length=16384")
flops_analysis(
        d_model, d_ff, num_layers, 
        16384, vocab_size, 
)

GPT-2 XL with context_length=1024

MHA layer: 27682406400 FLOPs
Swiglu layer: 42152755200 FLOPs
Total Transformer Block: 3516769894400 FLOPs

MHA/Swiglu: 0.6567164179104478

Swiglu/MHA: 1.5227272727272727

GPT-2 XL with context_length=16384

MHA layer: 2053531238400 FLOPs
Swiglu layer: 674444083200 FLOPs
Total Transformer Block: 133577729638400 FLOPs

MHA/Swiglu: 3.044776119402985

Swiglu/MHA: 0.3284313725490196



In [8]:
133577729638400 / 3516769894400

37.983073573026545

## AdamW accounting

### (a) Peak memory of running AdamW

Everything is `float32`, so every tensor costs **4 bytes per element**. Denote the hyperparameters as $V$ (vocab_size), $L$ (context_length), $n$ (num_layers), $d$ (d_model), $h$ (num_heads), $B$ (batch_size), and use $d_{ff} = \tfrac{8}{3}d$.

There are four consumers of memory: **parameters**, **gradients**, **optimizer state**, and **activations**.

#### Parameters
Reusing the count from the Transformer accounting (no weight tying):

| Component | Params |
|---|---|
| Token embedding | $Vd$ |
| Per block: 2 × RMSNorm | $2d$ |
| Per block: MHA ($Q,K,V,O$) | $4d^2$ |
| Per block: SwiGLU ($W_1, W_2, W_3$) | $3 d_{ff} d = 8d^2$ |
| Final RMSNorm | $d$ |
| Output embedding (LM head) | $Vd$ |

$$P = 2Vd + n\,(12d^2 + 2d) + d$$

#### Gradients
One gradient per parameter ⇒ exactly $P$ floats.

#### Optimizer state
AdamW keeps the first moment $m$ **and** the second moment $v$ ⇒ $2P$ floats.

#### Activations (saved for backward), counting only the listed components

**Per Transformer block:**

| Component | Shape | Floats |
|---|---|---|
| 2 × RMSNorm | $(B,L,d)$ | $2BLd$ |
| QKV projections | $(B,L,d)\times3$ | $3BLd$ |
| $QK^\top$ | $(B,h,L,L)$ | $BhL^2$ |
| softmax | $(B,h,L,L)$ | $BhL^2$ |
| weighted sum of values | $(B,L,d)$ | $BLd$ |
| output projection | $(B,L,d)$ | $BLd$ |
| $W_1$, $W_3$, SiLU, element-wise product | $(B,L,d_{ff})\times4$ | $4BLd_{ff}$ |
| $W_2$ | $(B,L,d)$ | $BLd$ |

Per block $= 8BLd + 4BLd_{ff} + 2BhL^2 \;\overset{d_{ff}=\frac{8}{3}d}{=}\; \tfrac{56}{3}BLd + 2BhL^2$.

**Outside the blocks:** final RMSNorm $BLd$ + output embedding (logits) $BLV$ + cross-entropy on logits $BLV$.

$$A = n\!\left(\tfrac{56}{3}BLd + 2BhL^2\right) + BLd + 2BLV$$

#### Total peak memory (bytes)

$$\text{Mem} = \underbrace{4P}_{\text{params}} + \underbrace{4P}_{\text{grads}} + \underbrace{8P}_{\text{opt state}} + \underbrace{4A}_{\text{activations}} = 16P + 4A$$


In [9]:
BYTES_PER_FLOAT32 = 4
GiB = 1024 ** 3


def num_parameters(d_model: int, num_layers: int, vocab_size: int) -> int:
    # d_ff = 8/3 * d_model  =>  MHA (4 d^2) + SwiGLU (3 * d_ff * d = 8 d^2) = 12 d^2 per block
    per_block = 12 * d_model ** 2 + 2 * d_model
    return 2 * vocab_size * d_model + num_layers * per_block + d_model


def num_activations(
    batch_size: int, d_model: int, num_layers: int, num_heads: int,
    context_length: int, vocab_size: int,
) -> int:
    B, L, d, h = batch_size, context_length, d_model, num_heads
    d_ff = 8 * d / 3

    rmsnorm = 2 * B * L * d                       # 2 RMSNorm
    mha = 3 * B * L * d + 2 * B * h * L ** 2 + 2 * B * L * d   # QKV, QK^T+softmax, weighted sum + O proj
    ffn = 4 * B * L * d_ff + B * L * d            # W1, W3, SiLU, product (d_ff) + W2 (d)
    per_block = rmsnorm + mha + ffn

    final_rmsnorm = B * L * d
    output_embedding = B * L * vocab_size
    cross_entropy = B * L * vocab_size
    return num_layers * per_block + final_rmsnorm + output_embedding + cross_entropy


def peak_memory_bytes(
    batch_size: int, d_model: int, num_layers: int, num_heads: int,
    context_length: int, vocab_size: int,
) -> dict:
    P = num_parameters(d_model, num_layers, vocab_size)
    A = num_activations(batch_size, d_model, num_layers, num_heads, context_length, vocab_size)
    return {
        "parameters": 4 * P,
        "gradients": 4 * P,
        "optimizer_state": 8 * P,   # m and v
        "activations": 4 * A,
        "total": 16 * P + 4 * A,
    }


mem = peak_memory_bytes(1, d_model, num_layers, num_heads, context_length, vocab_size)
for k, v in mem.items():
    print(f"{k:>16}: {v / GiB:8.3f} GiB")


      parameters:    6.093 GiB
       gradients:    6.093 GiB
 optimizer_state:   12.186 GiB
     activations:   15.233 GiB
           total:   39.605 GiB


### (b) Instantiate for GPT-2 XL

The parameter / gradient / optimizer memory does **not** depend on the batch size — only the activations do. So the total memory is affine in `batch_size`:

$$\text{Mem}(B) = a \cdot B + b$$

where $b = 16P$ (params + grads + optimizer state) and $a = 4 \cdot (A/B)$ (activations per batch element). I compute $a$ and $b$ below and solve $a \cdot B + b \le 80\,\text{GiB}$ for the maximum integer batch size.


In [10]:
import math

# GPT-2 XL: V=50257, L=1024, n=48, d=1600, h=25  (d_ff = 8/3 * d_model)
b_bytes = 16 * num_parameters(d_model, num_layers, vocab_size)                       # batch-independent part
a_bytes = 4 * num_activations(1, d_model, num_layers, num_heads, context_length, vocab_size)  # per batch element

a_gib = a_bytes / GiB
b_gib = b_bytes / GiB
print(f"Mem(B) = {a_gib:.4f} * batch_size + {b_gib:.4f}  (GiB)")
print(f"       = {a_bytes} * batch_size + {b_bytes}  (bytes)")

BUDGET_GIB = 80
max_batch = math.floor((BUDGET_GIB - b_gib) / a_gib)
print(f"\nMax batch size within {BUDGET_GIB} GiB: {max_batch}")
print(f"Check: Mem({max_batch}) = {a_gib * max_batch + b_gib:.3f} GiB, "
      f"Mem({max_batch + 1}) = {a_gib * (max_batch + 1) + b_gib:.3f} GiB")


Mem(B) = 15.2333 * batch_size + 24.3714  (GiB)
       = 16356614144.0 * batch_size + 26168601600  (bytes)

Max batch size within 80 GiB: 3
Check: Mem(3) = 70.071 GiB, Mem(4) = 85.305 GiB


### (c) FLOPs of one AdamW step

One step = **forward pass** + **backward pass** + **optimizer update**.

* **Forward pass** matmul FLOPs (for batch $B$, from the Transformer accounting):
$$F_{\text{fwd}} = B\left[\,n\,(24\,d^2 L + 4\,d L^2) + 2\,d V L\,\right]$$
(per layer: $8d^2L$ for QKVO + $4dL^2$ for attention + $16d^2L$ for SwiGLU $=24d^2L+4dL^2$; plus $2dVL$ for the LM head.)

* **Backward pass:** following Kaplan/Hoffmann, costs **twice** the forward pass: $2F_{\text{fwd}}$.

* **AdamW update:** purely element-wise over the $P$ parameters (a handful of mults/adds each), so it is $\mathcal{O}(P)$ — on the order of $\sim\!10P$. Since $F_{\text{fwd}} \approx 2 P \cdot (BL) \gg P$, the optimizer step is **negligible**.

$$\boxed{F_{\text{step}} \approx 3\,F_{\text{fwd}} + cP \approx 3\,F_{\text{fwd}}}$$


### (d) Training time for GPT-2 XL

Train for $400\text{K}$ steps with `batch_size = 1024` on one H100.

* Per step: $F_{\text{step}} = 3 F_{\text{fwd}}$ (forward + 2× backward, optimizer negligible).
* Total: $F_{\text{total}} = \text{steps} \cdot F_{\text{step}}$.
* H100 peak (TF32) $= 495\ \text{TFLOP/s}$, at $50\%$ MFU the effective throughput is $0.5 \times 495 = 247.5\ \text{TFLOP/s}$.
* Time $= F_{\text{total}} / (\text{effective throughput})$.


In [11]:
def forward_flops(batch_size, d_model, num_layers, context_length, vocab_size):
    B, d, n, L, V = batch_size, d_model, num_layers, context_length, vocab_size
    per_layer = 24 * d ** 2 * L + 4 * d * L ** 2          # QKVO + attention + SwiGLU (d_ff = 8/3 d)
    head = 2 * d * V * L                                  # output embedding / LM head
    return B * (n * per_layer + head)

STEPS = 400_000
BATCH = 1024
PEAK_FLOPS = 495e12        # H100 TF32
MFU = 0.50

f_fwd = forward_flops(BATCH, d_model, num_layers, context_length, vocab_size)
f_step = 3 * f_fwd                       # forward + 2x backward
f_total = STEPS * f_step

effective = PEAK_FLOPS * MFU
seconds = f_total / effective
hours = seconds / 3600

print(f"Forward FLOPs / step : {f_fwd:.3e}")
print(f"Total training FLOPs : {f_total:.3e}")
print(f"Effective throughput : {effective:.3e} FLOP/s")
print(f"\nTraining time: {hours:,.0f} hours  (~{hours / 24:,.0f} days)")


Forward FLOPs / step : 3.591e+15
Total training FLOPs : 4.309e+21
Effective throughput : 2.475e+14 FLOP/s

Training time: 4,836 hours  (~202 days)
